In [4]:
import os

In [5]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project\\research'

In [6]:
os.chdir("../")

In [7]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project'

In [8]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path          # Base folder to store validation artifacts
    raw_data_file: Path     # CSV file produced by Data Ingestion
    status_file: Path       # Path to save validation status (success/fail)
    all_schema: dict        # Dictionary of expected columns and types


In [9]:
from WineQuality_Project.constants import *
from WineQuality_Project.utils.common import read_yaml,create_directories

In [12]:
class ConfigManager:
    def __init__(self,
                 config_filepath: Path = CONFIG_FILE_PATH,
                 params_filepath: Path = PARAMS_FILE_PATH,
                 schema_filepath: Path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([Path(self.config['artifact_root'])])

   

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config['data_validation']
        create_directories([Path(config['root_dir'])])
        return DataValidationConfig(
            root_dir=Path(config['root_dir']),
            raw_data_file=Path(config['raw_data_file']),
            status_file=Path(config['status_file']),
            all_schema=self.schema
        )

In [20]:
import pandas as pd
import logging
from pathlib import Path

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config
        self.schema = self.config.all_schema

    def _write_status(self, message: str, success=True):
        self.config.status_file.parent.mkdir(parents=True, exist_ok=True)
        with open(self.config.status_file, 'w') as f:
            if success:
                f.write(f"Validation Status: SUCCESS ✅\n{message}\n")
            else:
                f.write(f"Validation Status: FAILED ❌\n{message}\n")
        logging.info(f"Status written to {self.config.status_file}")

    def validate_file_existence(self):
        if not self.config.raw_data_file.exists():
            msg = f"File not found: {self.config.raw_data_file}"
            self._write_status(msg, success=False)
            raise FileNotFoundError(msg)
        logging.info(f"File exists: {self.config.raw_data_file}")

    def normalize_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Replace spaces with underscores and strip column names."""
        df.columns = [col.strip().replace(" ", "_") for col in df.columns]
        return df

    def validate_columns(self, df: pd.DataFrame):
        df = self.normalize_columns(df)
        expected_columns = list(self.schema['columns'].keys())
        csv_columns = list(df.columns)

        missing_columns = [col for col in expected_columns if col not in csv_columns]
        extra_columns = [col for col in csv_columns if col not in expected_columns]

        if missing_columns:
            msg = f"Missing columns: {missing_columns}"
            self._write_status(msg, success=False)
            raise ValueError(msg)

        logging.info(f"All required columns exist. Extra columns ignored: {extra_columns}")
        return df

    def validate_missing_values(self, df: pd.DataFrame):
        missing_count = df.isnull().sum().sum()
        if missing_count > 0:
            msg = f"Dataset contains {missing_count} missing values"
            self._write_status(msg, success=False)
            raise ValueError(msg)
        logging.info("No missing values found.")

    def initiate_data_validation(self) -> pd.DataFrame:
        try:
            self.validate_file_existence()
            df = pd.read_csv(self.config.raw_data_file)
            logging.info(f"CSV loaded: {self.config.raw_data_file}")
            df = self.validate_columns(df)
            self.validate_missing_values(df)
            self._write_status("Data validation completed successfully ✅", success=True)
            return df
        except Exception as e:
            logging.error(f"Data validation failed: {e}")
            raise e


In [21]:
try:
    config=ConfigManager()
    data_validation_config=config.get_data_validation_config()
    data_validation=DataValidation(config=data_validation_config)
    data=data_validation.initiate_data_validation()
except Exception as e:
    raise

[2025-12-11 23:26:45] [INFO] WineQualityLogger - YAML file: config\config.yml loaded successfully
INFO:WineQualityLogger:YAML file: config\config.yml loaded successfully
[2025-12-11 23:26:46] [INFO] WineQualityLogger - YAML file: params.yaml loaded successfully
INFO:WineQualityLogger:YAML file: params.yaml loaded successfully
[2025-12-11 23:26:46] [INFO] WineQualityLogger - YAML file: schema.yaml loaded successfully
INFO:WineQualityLogger:YAML file: schema.yaml loaded successfully
[2025-12-11 23:26:46] [INFO] WineQualityLogger - Directory created at: artifacts
INFO:WineQualityLogger:Directory created at: artifacts
[2025-12-11 23:26:46] [INFO] WineQualityLogger - Directory created at: artifacts\data_validation
INFO:WineQualityLogger:Directory created at: artifacts\data_validation
